<a href="https://colab.research.google.com/github/aarti-311/AI-ML-Labs/blob/main/LAB_7_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB 7 : APPLY ALL ALGO ON THE STOCK MARKET DATASET

In [ ]:
#Importing Libraries
import pandas as pd
import numpy as np
import nltk
import re
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import classification_report
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
# Reading the CSV file using pandas
df= pd.read_csv('Stock News Dataset.csv', encoding= 'unicode_escape')
# Displaying the first 10 rows of the dataframe
df.head()

,Date,Label,Top1,Top2,Top3,Top4,Top5,Top6,Top7,Top8,...,Top16,Top17,Top18,Top19,Top20,Top21,Top22,Top23,Top24,Top25
0,2000-01-03,0,A 'hindrance to operations': extracts from the...,Scorecard,Hughes' instant hit buoys Blues,Jack gets his skates on at ice-cold Alex,Chaos as Maracana builds up for United,Depleted Leicester prevail as Elliott spoils E...,Hungry Spurs sense rich pickings,Gunners so wide of an easy target,...,Flintoff injury piles on woe for England,Hunters threaten Jospin with new battle of the...,Kohl's successor drawn into scandal,The difference between men and women,"Sara Denver, nurse turned solicitor",Diana's landmine crusade put Tories in a panic,Yeltsin's resignation caught opposition flat-f...,Russian roulette,Sold out,Recovering a title
1,2000-01-04,0,Scorecard,The best lake scene,Leader: German sleaze inquiry,"Cheerio, boyo",The main recommendations,Has Cubie killed fees?,Has Cubie killed fees?,Has Cubie killed fees?,...,On the critical list,The timing of their lives,Dear doctor,Irish court halts IRA man's extradition to Nor...,Burundi peace initiative fades after rebels re...,PE points the way forward to the ECB,Campaigners keep up pressure on Nazi war crime...,Jane Ratcliffe,Yet more things you wouldn't know without the ...,Millennium bug fails to bite
2,2000-01-05,0,Coventry caught on counter by Flo,United's rivals on the road to Rio,Thatcher issues defence before trial by video,Police help Smith lay down the law at Everton,Tale of Trautmann bears two more retellings,England on the rack,Pakistan retaliate with call for video of Walsh,Cullinan continues his Cape monopoly,...,South Melbourne (Australia),Necaxa (Mexico),Real Madrid (Spain),Raja Casablanca (Morocco),Corinthians (Brazil),Tony's pet project,Al Nassr (Saudi Arabia),Ideal Holmes show,Pinochet leaves hospital after tests,Useful links
3,2000-01-06,1,Pilgrim knows how to progress,Thatcher facing ban,McIlroy calls for Irish fighting spirit,Leicester bin stadium blueprint,United braced for Mexican wave,"Auntie back in fashion, even if the dress look...",Shoaib appeal goes to the top,Hussain hurt by 'shambles' but lays blame on e...,...,Putin admits Yeltsin quit to give him a head s...,BBC worst hit as digital TV begins to bite,How much can you pay for...,Christmas glitches,"Upending a table, Chopping a line and Scoring ...","Scientific evidence 'unreliable', defence claims",Fusco wins judicial review in extradition case,Rebels thwart Russian advance,Blair orders shake-up of failing NHS,Lessons of law's hard heart
4,2000-01-07,1,Hitches and Horlocks,Beckham off but United survive,Breast cancer screening,Alan Parker,Guardian readers: are you all whingers?,Hollywood Beyond,Ashes and diamonds,Whingers - a formidable minority,...,Most everywhere: UDIs,Most wanted: Chloe lunettes,Return of the cane 'completely off the agenda',From Sleepy Hollow to Greeneland,Blunkett outlines vision for over 11s,"Embattled Dobson attacks 'play now, pay later'...",Doom and the Dome,What is the north-south divide?,Aitken released from jail,Gone aloft


In [ ]:
#Remove any null values from the dataset
df.dropna(inplace=True)

In [ ]:
# Using applymap(str) and dictionary notation
data = pd.DataFrame({
    'text': df.iloc[:, 2:].applymap(str).apply(' '.join, axis=1),
    'Label': df['Label']
})
data.head(3)

,text,Label
0,A 'hindrance to operations': extracts from the...,0
1,Scorecard The best lake scene Leader: German s...,0
2,Coventry caught on counter by Flo United's riv...,0


In [ ]:
pattern = r'\[[0-9]]*\]|\(.*?\)]|\d+|\s+|[^\w\s]'
data['text'] = data['text'].str.replace(pattern, ' ').str.lower()
data

<ipython-input-78-61be7346a002>:2: FutureWarning: The default value of regex will change from True to False in a future version.
  data['text'] = data['text'].str.replace(pattern, ' ').str.lower()


,text,Label
0,a hindrance to operations extracts from the...,0
1,scorecard the best lake scene leader german s...,0
2,coventry caught on counter by flo united s riv...,0
3,pilgrim knows how to progress thatcher facing ...,1
4,hitches and horlocks beckham off but united su...,1
...,...,...
4257,barclays and rbs shares suspended from trading...,0
4258,scientists to australia if you want to sa...,1
4259,explosion at airport in istanbul yemeni former...,1
4260,jamaica proposes marijuana dispensers for tour...,1


In [ ]:
stop_words = set(stopwords.words('english'))

data['text'] = data['text'].apply(lambda x: ' '.join([word for word in word_tokenize(x) if word.lower() not in stop_words]))
data

,text,Label
0,hindrance operations extracts leaked reports s...,0
1,scorecard best lake scene leader german sleaze...,0
2,coventry caught counter flo united rivals road...,0
3,pilgrim knows progress thatcher facing ban mci...,1
4,hitches horlocks beckham united survive breast...,1
...,...,...
4257,barclays rbs shares suspended trading tanking ...,0
4258,scientists australia want save great barrier r...,1
4259,explosion airport istanbul yemeni former presi...,1
4260,jamaica proposes marijuana dispensers tourists...,1


In [ ]:
def stemming(text):
    return ' '.join([PorterStemmer().stem(word) for word in word_tokenize(text)])

data['text'] = data['text'].apply(stemming)

In [ ]:
data

,text,Label
0,hindranc oper extract leak report scorecard hu...,0
1,scorecard best lake scene leader german sleaz ...,0
2,coventri caught counter flo unit rival road ri...,0
3,pilgrim know progress thatcher face ban mcilro...,1
4,hitch horlock beckham unit surviv breast cance...,1
...,...,...
4257,barclay rb share suspend trade tank pope say c...,0
4258,scientist australia want save great barrier re...,1
4259,explos airport istanbul yemeni former presid t...,1
4260,jamaica propos marijuana dispens tourist airpo...,1


In [ ]:
def lemmatization(text):
    return ' '.join([WordNetLemmatizer().lemmatize(word) for word in word_tokenize(text)])

data['text'] = data['text'].apply(lemmatization)

In [ ]:
data

,text,Label
0,hindranc oper extract leak report scorecard hu...,0
1,scorecard best lake scene leader german sleaz ...,0
2,coventri caught counter flo unit rival road ri...,0
3,pilgrim know progress thatcher face ban mcilro...,1
4,hitch horlock beckham unit surviv breast cance...,1
...,...,...
4257,barclay rb share suspend trade tank pope say c...,0
4258,scientist australia want save great barrier re...,1
4259,explos airport istanbul yemeni former presid t...,1
4260,jamaica propos marijuana dispens tourist airpo...,1


In [ ]:
x = data['text']
y = data['Label']
train_data, test_data, train_labels, test_labels = train_test_split(x, y, test_size=0.2, random_state=1)

In [ ]:
#BOW
cv = CountVectorizer(max_features=1500)
x_train_cv  = cv.fit_transform(train_data)
x_test_cv = cv.transform(test_data)

In [ ]:
cv.shape()

NameError: ignored

In [ ]:
#Random forest Classifier
rf= RandomForestClassifier(max_depth=2, random_state=0)
rf.fit(x_train_cv, train_labels)
# Predicting the labels of test and train data
y_pred_test_cv = rf.predict(x_test_cv)
y_pred_train_cv = rf.predict(x_train_cv)


In [ ]:
print(classification_report(test_labels, y_pred_test_cv))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       362
           1       0.55      1.00      0.71       439

    accuracy                           0.55       801
   macro avg       0.27      0.50      0.35       801
weighted avg       0.30      0.55      0.39       801



/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
# Initialize a TF-IDF vectorizer with a maximum of 1500 features
tfidf = TfidfVectorizer(max_features=1500)

# Fit and transform the training data using the vectorizer
x_train_tfidf = tfidf.fit_transform(train_data)

# Transform the test data using the trained vectorizer
x_test_tfidf = tfidf.transform(test_data)


In [ ]:
rf= RandomForestClassifier(max_depth=2, random_state=0)
rf.fit(x_train_tfidf, train_labels)
# Predicting the labels of test and train data
y_pred_test_tfidf = rf.predict(x_test_tfidf)
y_pred_train_tfidf = rf.predict(x_train_tfidf)

In [ ]:
print(classification_report(test_labels, y_pred_test_tfidf))

              precision    recall  f1-score   support

           0       1.00      0.00      0.01       362
           1       0.55      1.00      0.71       439

    accuracy                           0.55       801
   macro avg       0.77      0.50      0.36       801
weighted avg       0.75      0.55      0.39       801



In [ ]:
#WORD 2 VEC
# Tokenize the text data
tokenized_data = [word_tokenize(text) for text in data['text']]

# Train the word2vec model
model = Word2Vec(tokenized_data, min_count=1)
vocabulary= model.wv.vocab

# save the word2vec model to a file
model.save("word2vec.model")

# Get the vector representation of a word
word_vector = model['jamaica']
word_vector

<ipython-input-91-24d218055bb0>:13: DeprecationWarning: Call to deprecated `__getitem__` (Method will be removed in 4.0.0, use self.wv.__getitem__() instead).
  word_vector = model['jamaica']


array([-0.0464042 ,  0.0641266 , -0.19547598, -0.19013545,  0.09696428,
       -0.06690282, -0.16121519, -0.12977675,  0.07560878, -0.03596312,
        0.0680895 , -0.08130886,  0.22762057, -0.22533299,  0.04611552,
       -0.17427497,  0.2524107 , -0.03071862, -0.02584545,  0.04003998,
       -0.10774215, -0.18317972,  0.15448461,  0.12409176,  0.09658539,
       -0.06546776,  0.0109247 ,  0.10483123,  0.10202297,  0.06611395,
       -0.02095138,  0.19571455,  0.12789105,  0.10390981,  0.12515597,
       -0.07455764, -0.09631743, -0.01246476, -0.13009045,  0.00666029,
        0.13210307, -0.29602435, -0.10903593, -0.02071914, -0.02598305,
        0.1009202 ,  0.04100879, -0.08612782, -0.22273561,  0.08401129,
       -0.01352746, -0.02926155,  0.05381245,  0.18337667, -0.06967071,
        0.28666168,  0.14892936, -0.09706899, -0.02242524,  0.1306427 ,
       -0.21774453, -0.02967951,  0.20315741, -0.14184149,  0.17398249,
        0.05973834, -0.09383323,  0.12977824, -0.27770197, -0.10

In [ ]:
print(model.wv.similarity(w1="airport", w2="australia"))

0.7272913


In [ ]:
similar= model.wv.most_similar('progress')
similar

[('abyss', 0.9971423745155334),
 ('everyon', 0.9969073534011841),
 ('byte', 0.9968003034591675),
 ('imagin', 0.9966319799423218),
 ('cake', 0.996562123298645),
 ('grief', 0.9964022636413574),
 ('practic', 0.9963406324386597),
 ('whatev', 0.9963365197181702),
 ('nice', 0.9963316917419434),
 ('undo', 0.9960781931877136)]